# Ringkasan Struktur Data ISPU DKI Jakarta

Notebook ini memaparkan struktur awal dataset **ISPU DKI Jakarta** yang mencakup:

1. jumlah baris dan kolom,
2. nama kolom dan tipe data setiap kolom,
3. periode waktu dataset,
4. cakupan stasiun pemantauan,
5. catatan awal terkait struktur data yang perlu diperhatikan sebelum analisis lanjutan.


## 1. Import Library dan Load Dataset

Dataset dibaca dari file `ispu_jakarta_clean.csv`. Apabila notebook dijalankan di luar environment ini, pastikan file CSV berada dalam folder yang sama dengan notebook.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "ispu_jakarta_clean.csv"

ALTERNATIVE_DATA_PATHS = [
    DATA_PATH,
    PROJECT_ROOT / "ispu_jakarta_analysis.csv",
    Path.cwd() / "outputs" / "ispu_jakarta_analysis.csv",
    Path.cwd() / "ispu_jakarta_analysis.csv",
]

df = pd.read_csv(DATA_PATH)

display(df.head())

,tanggal,periode_data,pm10,pm25,so2,co,o3,no2,max,critical,categori,stasiun
0,2010-01-01,201001.0,60.0,NaN,4.0,73.0,27.0,14.0,73.0,CO,SEDANG,DKI1 Bunderan HI
1,2010-01-02,201001.0,32.0,NaN,2.0,16.0,33.0,9.0,33.0,O3,BAIK,DKI1 Bunderan HI
2,2010-01-03,201001.0,27.0,NaN,2.0,19.0,20.0,9.0,27.0,PM10,BAIK,DKI1 Bunderan HI
3,2010-01-04,201001.0,22.0,NaN,2.0,16.0,15.0,6.0,22.0,PM10,BAIK,DKI1 Bunderan HI
4,2010-01-05,201001.0,25.0,NaN,2.0,17.0,15.0,8.0,25.0,PM10,BAIK,DKI1 Bunderan HI


## 2. Ringkasan Dimensi Dataset

Bagian ini menampilkan jumlah baris dan kolom pada dataset.

In [3]:
jumlah_baris, jumlah_kolom = df.shape

summary_shape = pd.DataFrame({
    "Metrik": ["Jumlah baris", "Jumlah kolom"],
    "Nilai": [jumlah_baris, jumlah_kolom]
})

display(summary_shape)

,Metrik,Nilai
0,Jumlah baris,14577
1,Jumlah kolom,12


**Interpretasi singkat:** dataset terdiri dari **14.577 baris** dan **12 kolom**. Setiap baris merepresentasikan catatan ISPU harian pada suatu stasiun pemantauan.

## 3. Nama Kolom dan Tipe Data

Tabel berikut menunjukkan nama kolom, tipe data awal ketika CSV dibaca oleh pandas, jumlah nilai terisi, jumlah nilai kosong, dan contoh nilai pada setiap kolom.

In [4]:
column_info = pd.DataFrame({
    "No": range(1, len(df.columns) + 1),
    "Nama Kolom": df.columns,
    "Tipe Data Awal": [str(dtype) for dtype in df.dtypes],
    "Jumlah Non-Null": [df[col].notna().sum() for col in df.columns],
    "Jumlah Missing": [df[col].isna().sum() for col in df.columns],
    "Contoh Nilai": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})

display(column_info)

,No,Nama Kolom,Tipe Data Awal,Jumlah Non-Null,Jumlah Missing,Contoh Nilai
0,1,tanggal,str,14577,0,2010-01-01
1,2,periode_data,float64,14576,1,201001.0
2,3,pm10,float64,14576,1,60.0
3,4,pm25,float64,5517,9060,51.0
4,5,so2,float64,14573,4,4.0
5,6,co,float64,14576,1,73.0
6,7,o3,float64,14576,1,27.0
7,8,no2,float64,14576,1,14.0
8,9,max,float64,14576,1,73.0
9,10,critical,str,14576,1,CO


### Keterangan Variabel

| Kolom | Keterangan Singkat |
|---|---|
| `tanggal` | Tanggal pengamatan ISPU. |
| `periode_data` | Periode data dalam format tahun-bulan, misalnya `202311`. |
| `pm10` | Nilai indeks parameter partikulat PM10. |
| `pm25` | Nilai indeks parameter partikulat PM2.5. |
| `so2` | Nilai indeks parameter sulfur dioksida. |
| `co` | Nilai indeks parameter karbon monoksida. |
| `o3` | Nilai indeks parameter ozon. |
| `no2` | Nilai indeks parameter nitrogen dioksida. |
| `max` | Nilai indeks maksimum dari parameter pencemar pada hari tersebut. |
| `critical` | Parameter pencemar dominan/kritis yang menjadi penentu nilai maksimum. |
| `categori` | Kategori kualitas udara berdasarkan nilai ISPU. |
| `stasiun` | Nama stasiun pemantauan kualitas udara. |


## 4. Konversi Tipe Tanggal untuk Analisis Periode

Kolom `tanggal` dibaca sebagai `object` karena berasal dari CSV. Untuk kebutuhan analisis periode, kolom tersebut dikonversi menjadi tipe `datetime`. Nilai yang gagal dikonversi akan menjadi `NaT` dan perlu diperiksa.

In [5]:
df_analysis = df.copy()
df_analysis["tanggal_dt"] = pd.to_datetime(df_analysis["tanggal"], errors="coerce")

conversion_summary = pd.DataFrame({
    "Kolom": ["tanggal", "tanggal_dt"],
    "Tipe Data": [str(df_analysis["tanggal"].dtype), str(df_analysis["tanggal_dt"].dtype)],
    "Keterangan": [
        "Tipe awal dari CSV",
        "Tipe hasil konversi untuk analisis waktu"
    ]
})

display(conversion_summary)
print(f"Jumlah tanggal yang gagal dikonversi: {df_analysis['tanggal_dt'].isna().sum()}")

,Kolom,Tipe Data,Keterangan
0,tanggal,str,Tipe awal dari CSV
1,tanggal_dt,datetime64[us],Tipe hasil konversi untuk analisis waktu


Jumlah tanggal yang gagal dikonversi: 1


## 5. Periode Waktu Dataset

Periode waktu dihitung berdasarkan kolom `tanggal` yang sudah dikonversi menjadi `datetime`.

In [6]:
valid_date = df_analysis["tanggal_dt"].notna()

periode_summary = pd.DataFrame({
    "Metrik": [
        "Tanggal awal",
        "Tanggal akhir",
        "Jumlah tanggal unik",
        "Jumlah bulan unik",
        "Periode data awal",
        "Periode data akhir"
    ],
    "Nilai": [
        df_analysis.loc[valid_date, "tanggal_dt"].min().date(),
        df_analysis.loc[valid_date, "tanggal_dt"].max().date(),
        df_analysis.loc[valid_date, "tanggal_dt"].nunique(),
        df_analysis.loc[valid_date, "tanggal_dt"].dt.to_period("M").nunique(),
        int(df_analysis["periode_data"].min()) if df_analysis["periode_data"].notna().any() else None,
        int(df_analysis["periode_data"].max()) if df_analysis["periode_data"].notna().any() else None
    ]
})

display(periode_summary)

,Metrik,Nilai
0,Tanggal awal,2010-01-01
1,Tanggal akhir,2023-11-30
2,Jumlah tanggal unik,4773
3,Jumlah bulan unik,166
4,Periode data awal,201001
5,Periode data akhir,202311


**Interpretasi singkat:** rentang waktu valid dalam dataset adalah **1 Januari 2010 sampai 30 November 2023**. Dataset mencakup **4.773 tanggal unik** dan **166 bulan unik** berdasarkan kolom tanggal.

## 6. Cakupan Stasiun Pemantauan

Bagian ini menampilkan jumlah stasiun, daftar stasiun, jumlah baris per stasiun, serta periode data yang tersedia pada masing-masing stasiun.

In [7]:
station_list = sorted(df_analysis["stasiun"].dropna().unique())

station_summary = pd.DataFrame({
    "Metrik": ["Jumlah stasiun unik", "Daftar stasiun"],
    "Nilai": [len(station_list), ", ".join(station_list)]
})

display(station_summary)

,Metrik,Nilai
0,Jumlah stasiun unik,5
1,Daftar stasiun,"DKI1 Bunderan HI, DKI2 Kelapa Gading, DKI3 Jag..."


In [8]:
station_coverage = (
    df_analysis
    .dropna(subset=["stasiun"])
    .groupby("stasiun")
    .agg(
        jumlah_baris=("tanggal", "size"),
        tanggal_awal=("tanggal_dt", "min"),
        tanggal_akhir=("tanggal_dt", "max"),
        jumlah_tanggal_unik=("tanggal_dt", "nunique")
    )
    .reset_index()
)

station_coverage["tanggal_awal"] = station_coverage["tanggal_awal"].dt.date
station_coverage["tanggal_akhir"] = station_coverage["tanggal_akhir"].dt.date

display(station_coverage)

,stasiun,jumlah_baris,tanggal_awal,tanggal_akhir,jumlah_tanggal_unik
0,DKI1 Bunderan HI,2940,2010-01-01,2023-11-30,2891
1,DKI2 Kelapa Gading,2881,2010-11-04,2023-11-30,2792
2,DKI3 Jagakarsa,2929,2010-11-07,2023-11-30,2837
3,DKI4 Lubang Buaya,3345,2010-11-04,2023-11-30,3143
4,DKI5 Kebon Jeruk,2481,2012-11-27,2023-11-30,2417


**Interpretasi singkat:** dataset mencakup **5 stasiun pemantauan**, yaitu:

1. DKI1 Bunderan HI,
2. DKI2 Kelapa Gading,
3. DKI3 Jagakarsa,
4. DKI4 Lubang Buaya,
5. DKI5 Kebon Jeruk.

Cakupan tanggal setiap stasiun tidak sepenuhnya sama. Stasiun DKI1 memiliki data sejak 2010-01-01, sedangkan DKI5 mulai muncul pada 2012-11-27.

## 7. Catatan Awal Struktur Data

Bagian ini bukan tahap cleaning penuh, tetapi pengecekan awal untuk mengetahui apakah ada anomali struktur yang perlu diperhatikan.

In [9]:
invalid_date_rows = df_analysis[df_analysis["tanggal_dt"].isna()]

print(f"Jumlah baris dengan tanggal tidak valid: {len(invalid_date_rows)}")
if len(invalid_date_rows) > 0:
    display(invalid_date_rows)

Jumlah baris dengan tanggal tidak valid: 1


,tanggal,periode_data,pm10,pm25,so2,co,o3,no2,max,critical,categori,stasiun,tanggal_dt
8234,"2021-08-25,202108,"" "",81.0,44.0,10.0,33.0,11.0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [10]:
category_summary = df_analysis["categori"].value_counts(dropna=False).reset_index()
category_summary.columns = ["Kategori", "Jumlah Baris"]

display(category_summary)

,Kategori,Jumlah Baris
0,SEDANG,9731
1,BAIK,2398
2,TIDAK SEHAT,2258
3,SANGAT TIDAK SEHAT,187
4,BERBAHAYA,1
5,NaN,1
6,LUARBIASA,1


In [11]:
critical_summary = df_analysis["critical"].value_counts(dropna=False).reset_index()
critical_summary.columns = ["Parameter Critical", "Jumlah Baris"]

display(critical_summary)

,Parameter Critical,Jumlah Baris
0,O3,6643
1,PM10,3609
2,PM2.5,3483
3,CO,397
4,SO2,287
5,2,112
6,1,30
7,3,7
8,NO2,5
9,5,3


**Catatan awal:**

- Terdapat **1 baris** dengan format tanggal tidak valid karena satu baris CSV tampak masuk seluruhnya ke kolom `tanggal`.
- Kolom `critical` memiliki nilai non-standar seperti `1`, `2`, `3`, dan `5`, sehingga perlu distandarkan kembali apabila akan dipakai untuk analisis parameter pencemar dominan.
- Kolom `categori` memiliki nilai `LUARBIASA` yang tampak tidak konsisten dengan kategori ISPU umum, sehingga perlu divalidasi sebelum analisis final.
- Kolom `pm25` memiliki nilai kosong cukup banyak, kemungkinan karena periode pengukuran PM2.5 tidak tersedia penuh sejak awal periode data. Perlu diperlakukan hati-hati saat analisis tren jangka panjang.


## 8. Kesimpulan Ringkasan Struktur Data

Dataset ISPU DKI Jakarta memiliki struktur utama sebagai berikut:

| Aspek | Ringkasan |
|---|---|
| Jumlah baris | 14.577 baris |
| Jumlah kolom | 12 kolom |
| Periode waktu valid | 2010-01-01 sampai 2023-11-30 |
| Jumlah tanggal unik | 4.773 tanggal |
| Jumlah bulan unik | 166 bulan |
| Cakupan stasiun | 5 stasiun SPKU |
| Unit observasi | Catatan ISPU harian per stasiun |
| Catatan struktur | Ada 1 baris tanggal tidak valid dan beberapa nilai kategori/critical yang perlu distandarkan |

Notebook ini dapat menjadi bagian awal dari tahapan **pemahaman data** sebelum masuk ke proses validasi, data cleaning, eksplorasi data, dan pembangunan dashboard analitik.